05 Ground Truth Audit

Goal: combine the four ground-truth batches, add complexity/ambiguity/should-move labels, and create review queues.

In [30]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features import place_complexity, pin_ambiguity, should_move_rule
from src.metrics import task_aware_report, segmented_task_report

PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT, PROCESSED


(PosixPath('/Users/shivanibelambe/Pin-To-Place'),
 PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed'))

In [31]:
files = sorted(PROCESSED.glob("ground_truth_*.csv"))

files

[PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/ground_truth_0_999.csv'),
 PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/ground_truth_1000_1999.csv'),
 PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/ground_truth_2000_2999.csv'),
 PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/ground_truth_3000_3424.csv'),
 PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/ground_truth_combined.csv')]

In [32]:
df = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

df["place_complexity"] = df.apply(place_complexity, axis=1)
df["pin_ambiguity"] = df.apply(pin_ambiguity, axis=1)
df["should_move"] = df.apply(should_move_rule, axis=1)

df.to_csv(PROCESSED / "ground_truth_combined.csv", index=False)

df.shape

(6850, 35)

In [33]:
task_aware_report(df)

{'count': 6836,
 'mean_m': np.float64(2.69),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(20.02),
 'p95_m': np.float64(23.34),
 'max_m': np.float64(74.77),
 'pct_exact_no_move': np.float64(88.5),
 'pct_over_10m': np.float64(11.5),
 'pct_over_25m': np.float64(3.1),
 'pct_over_50m': np.float64(0.0)}

In [34]:
segmented_task_report(df, "tier_label")


,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,418,7.34,0.0,23.13,24.44,32.98,67.5,32.5,4.8,0.0,open_space
3,4600,3.29,0.0,21.65,24.36,74.77,86.0,14.0,4.2,0.0,standard_commercial
0,326,0.43,0.0,0.00,0.00,26.97,98.2,1.8,0.6,0.0,multi_tenant
1,1492,0.00,0.0,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [35]:
segmented_task_report(df, "place_complexity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,244,3.66,0.0,22.11,25.03,31.87,84.4,15.6,5.7,0.0,multi_tenant
0,604,4.37,0.0,22.46,24.15,33.65,81.1,18.9,4.0,0.0,complex
2,5988,2.48,0.0,18.57,23.19,74.77,89.4,10.6,2.9,0.0,simple


In [36]:
segmented_task_report(df, "pin_ambiguity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,4116,3.46,0.0,21.78,23.87,74.77,85.2,14.8,4.1,0.0,low
2,662,2.22,0.0,0.00,22.93,35.84,90.6,9.4,3.0,0.0,medium
0,2058,1.28,0.0,0.00,18.63,33.65,94.5,5.5,1.2,0.0,high


In [37]:
review_cols = [
    "id",
    "name",
    "category_primary",
    "region",
    "tier_label",
    "place_complexity",
    "pin_ambiguity",
    "gt_confidence",
    "offset_haversine_m",
    "should_move",
    "gt_reasoning",
]

high_offset = df[df["offset_haversine_m"] >= 30].sort_values(
    "offset_haversine_m",
    ascending=False,
)

high_offset[review_cols].head(75)

,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
6501,08f489c0a35ab63203e9b0cd0f7c2d30,Subway,sandwich_shop,TX,standard_commercial,simple,low,0.9,74.772429,True,The pin is placed at the main customer entranc...
3076,08f489c0a35ab63203e9b0cd0f7c2d30,Subway,sandwich_shop,TX,standard_commercial,simple,low,0.9,74.772429,True,The pin is placed at the main customer entranc...
869,08f2ab31ad9098c00350623fdcc1f6e6,Mi Bella Casa Furniture,furniture_store,OH,standard_commercial,simple,low,0.9,38.054800,True,The identified location is the main entrance o...
4294,08f2ab31ad9098c00350623fdcc1f6e6,Mi Bella Casa Furniture,furniture_store,OH,standard_commercial,simple,low,0.9,38.054800,True,The identified location is the main entrance o...
6332,08f2659a0cc6590803dd941c90df1da4,FairBridge Inn Express Memphis,hotel,TN,standard_commercial,simple,medium,0.9,35.839320,True,The identified location is the main entrance o...
2907,08f2659a0cc6590803dd941c90df1da4,FairBridge Inn Express Memphis,hotel,TN,standard_commercial,simple,medium,0.9,35.839320,True,The identified location is the main entrance o...
2039,08f264ab1318032303ad234d707ae6bf,Redbox,music_and_dvd_store,TN,standard_commercial,simple,low,0.9,34.562466,True,The pin is placed at the main customer entranc...
5464,08f264ab1318032303ad234d707ae6bf,Redbox,music_and_dvd_store,TN,standard_commercial,simple,low,0.9,34.562466,True,The pin is placed at the main customer entranc...
3558,08f2a98ba908c89403ed8847c6c2854b,Rc's Country Store & Fillin' Station,convenience_store,WV,standard_commercial,complex,high,0.9,33.648366,True,The main entrance is located on the street-fac...
133,08f2a98ba908c89403ed8847c6c2854b,Rc's Country Store & Fillin' Station,convenience_store,WV,standard_commercial,complex,high,0.9,33.648366,True,The main entrance is located on the street-fac...


In [38]:
low_confidence = df[df["gt_confidence"] < 0.6].sort_values(
    ["tier_label", "gt_confidence"],
    ascending=[True, True],
)

low_confidence[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
329,08f44c42e312cac103267191aa401ab4,Perfect Equipment Company,professional_services,GA,no_building,simple,high,0.2,0.0,False,No dedicated commercial structure is visible a...
2167,08f26c805ba944ce031b99b608e227ff,Sysarc Infomatix,software_development,TX,no_building,simple,high,0.2,0.0,False,No dedicated commercial structure is visible; ...
2459,08f26e582269229b03725c20585351e4,Terra Herring Photography,professional_services,KS,no_building,simple,high,0.2,0.0,False,No dedicated commercial structure is visible a...
3754,08f44c42e312cac103267191aa401ab4,Perfect Equipment Company,professional_services,GA,no_building,simple,high,0.2,0.0,False,No dedicated commercial structure is visible a...
5592,08f26c805ba944ce031b99b608e227ff,Sysarc Infomatix,software_development,TX,no_building,simple,high,0.2,0.0,False,No dedicated commercial structure is visible; ...
...,...,...,...,...,...,...,...,...,...,...,...
380,08f28347ac20c56903854b5b9b276da2,NTT Communications,software_development,CA,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
382,08f262eb18820b11038fa15113abce27,Hoglund Law,bankruptcy_law,MN,no_building,multi_tenant,high,0.3,0.0,False,No dedicated commercial structure is visible a...
390,08f26c188d8718ce03186d609403eaaa,Law Firm of Oklahoma,lawyer,OK,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
391,08f2a8b9b0aca98d037a18a32f529b30,Vaughan & Associates,professional_services,VA,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...


In [39]:
multi_tenant = df[df["tier_label"] == "multi_tenant"].sort_values(
    "offset_haversine_m",
    ascending=False,
)

multi_tenant[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
6156,08f441ad446d54200323a1a9871a5d0d,Pinch A Penny Pool Patio Spa,hot_tubs_and_pools,FL,multi_tenant,simple,low,0.9,26.973491,True,The specific unit for 'Pinch A Penny Pool Pati...
2731,08f441ad446d54200323a1a9871a5d0d,Pinch A Penny Pool Patio Spa,hot_tubs_and_pools,FL,multi_tenant,simple,low,0.9,26.973491,True,The specific unit for 'Pinch A Penny Pool Pati...
2698,08f26653a97ab63003e359ea728ee94e,Minute Clinic,shopping,IN,multi_tenant,simple,low,0.9,23.185469,True,The Minute Clinic is located in the strip mall...
6123,08f26653a97ab63003e359ea728ee94e,Minute Clinic,shopping,IN,multi_tenant,simple,low,0.9,23.185469,True,The Minute Clinic is located in the strip mall...
3539,08f2a84cf0035135038ee0c8f795caff,Gateway China Company,shopping,PA,multi_tenant,simple,low,0.9,20.034039,True,The pin is placed at the visible entrance of t...
...,...,...,...,...,...,...,...,...,...,...,...
4059,08f2a931994004f0039c70ef7f931106,Northgate Mall,shopping_center,OH,multi_tenant,complex,high,1.0,0.000000,False,The current pin is already at the correct unit...
4028,08f27404080136c103408eda82d4de21,Redbox,rental_kiosks,MI,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
4002,08f44ddcce80654a03e14bc7fdede62b,CVS Beauty,shopping,SC,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
3986,08f44dd18c73216b030211f1bfe568f2,Kay's Early Care and Education,day_care_preschool,SC,multi_tenant,simple,low,0.9,0.000000,False,The current pin is already at the correct unit...


In [40]:
should_move = df[df["should_move"]].sort_values(
    "offset_haversine_m",
    ascending=False,
)

should_move[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
3076,08f489c0a35ab63203e9b0cd0f7c2d30,Subway,sandwich_shop,TX,standard_commercial,simple,low,0.9,74.772429,True,The pin is placed at the main customer entranc...
6501,08f489c0a35ab63203e9b0cd0f7c2d30,Subway,sandwich_shop,TX,standard_commercial,simple,low,0.9,74.772429,True,The pin is placed at the main customer entranc...
869,08f2ab31ad9098c00350623fdcc1f6e6,Mi Bella Casa Furniture,furniture_store,OH,standard_commercial,simple,low,0.9,38.054800,True,The identified location is the main entrance o...
4294,08f2ab31ad9098c00350623fdcc1f6e6,Mi Bella Casa Furniture,furniture_store,OH,standard_commercial,simple,low,0.9,38.054800,True,The identified location is the main entrance o...
2907,08f2659a0cc6590803dd941c90df1da4,FairBridge Inn Express Memphis,hotel,TN,standard_commercial,simple,medium,0.9,35.839320,True,The identified location is the main entrance o...
...,...,...,...,...,...,...,...,...,...,...,...
525,08f44ccd2e31672e0310e0fac96580cf,Courthouse Tool & Tractor Rental,machine_and_tool_rentals,GA,standard_commercial,multi_tenant,medium,0.9,28.142093,True,The identified entrance is on the street-facin...
3950,08f44ccd2e31672e0310e0fac96580cf,Courthouse Tool & Tractor Rental,machine_and_tool_rentals,GA,standard_commercial,multi_tenant,medium,0.9,28.142093,True,The identified entrance is on the street-facin...
3093,08f44d075a9a3658036bc19f809d154f,CVS Pharmacy,pharmacy,SC,standard_commercial,simple,low,0.9,28.051978,True,The main customer entrance is located on the s...
6518,08f44d075a9a3658036bc19f809d154f,CVS Pharmacy,pharmacy,SC,standard_commercial,simple,low,0.9,28.051978,True,The main customer entrance is located on the s...


In [41]:
zero_offset_sample = (
    df[df["offset_haversine_m"] == 0]
    .sample(n=min(150, (df["offset_haversine_m"] == 0).sum()), random_state=42)
)

zero_offset_sample[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
1923,08f29a00c1b3406003a09d3daa92b40d,Baskin-Robbins,ice_cream_shop,CA,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the main custome...
3582,08f44dc41d1061b503c6c8a20c2caa4e,Rough Water Docks,construction_services,SC,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
4492,08f2a33076945d9903d177270b372474,Divines Fabric & Sewing Nook,fabric_store,RI,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the main custome...
4658,08f29a1d2e4258db03b7c4087eb8f2a9,Nora Lighting,lighting_store,CA,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the main custome...
3145,08f44d9806bb4a6903317861f734cac6,Clement Law Firm,lawyer,NC,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...
...,...,...,...,...,...,...,...,...,...,...,...
1766,08f275263230eb0503eb00ee6d80cf23,ADT Security Services,professional_services,MN,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
3017,08f26d152e72055003a8ef8141d48a85,Four States Recycling,garbage_collection_service,TX,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
1098,08f2ab576264830a032d0872e7b319c3,U Pick 6 Tap House,pub,PA,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the main custome...
6812,08f27434266e6b3103d9de9bb1fa3249,ALDI USA,supermarket,WI,standard_commercial,simple,low,1.0,0.0,False,The current pin is already located at the main...


In [42]:
outputs = {
    "review_high_offset.csv": high_offset,
    "review_low_confidence.csv": low_confidence,
    "review_multi_tenant.csv": multi_tenant,
    "review_should_move.csv": should_move,
    "review_zero_offset_sample.csv": zero_offset_sample,
}

for filename, frame in outputs.items():
    frame.to_csv(PROCESSED / filename, index=False)

list(outputs.keys())


['review_high_offset.csv',
 'review_low_confidence.csv',
 'review_multi_tenant.csv',
 'review_should_move.csv',
 'review_zero_offset_sample.csv']

In [43]:
summary_lines = []

summary_lines.append("Ground Truth Audit Summary")
summary_lines.append("")
summary_lines.append("Overall")
summary_lines.append(str(task_aware_report(df)))
summary_lines.append("")
summary_lines.append("By tier")
summary_lines.append(segmented_task_report(df, "tier_label").to_string(index=False))
summary_lines.append("")
summary_lines.append("By place complexity")
summary_lines.append(segmented_task_report(df, "place_complexity").to_string(index=False))
summary_lines.append("")
summary_lines.append("By ambiguity")
summary_lines.append(segmented_task_report(df, "pin_ambiguity").to_string(index=False))
summary_lines.append("")
summary_lines.append(f"High-offset review rows: {len(high_offset)}")
summary_lines.append(f"Low-confidence review rows: {len(low_confidence)}")
summary_lines.append(f"Multi-tenant review rows: {len(multi_tenant)}")
summary_lines.append(f"Should-move rows: {len(should_move)}")
summary_lines.append(f"Zero-offset sample rows: {len(zero_offset_sample)}")

summary_path = PROCESSED / "ground_truth_audit_summary.txt"
summary_path.write_text("\n".join(summary_lines))

summary_path


PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/ground_truth_audit_summary.txt')